<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Temperature_Celsius/Temperature_without_Iteration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

# =========================
# 1. Load datasets
# =========================
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Temperature_Celsius/temp_training_dataset.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Temperature_Celsius/temp_testing_dataset.csv")

# =========================
# 2. Features & Target
# =========================
features = [
    "uv_index",
    "latitude",
    "humidity",
    "pressure_mb",
    "air_quality_Ozone",
    "condition_text",
    "air_quality_Nitrogen_dioxide",
    "cloud",
    "visibility_km",
    "longitude",
    "air_quality_PM10",
    "wind_mph",
    "gust_mph",
    "wind_degree",
    "precip_mm"
]

target = "temperature_celsius"

# =========================
# 3. Keep needed columns
# =========================
train_df = train_df[features + [target]].copy()
test_df = test_df[features + [target]].copy()

# condition_text already encoded (force numeric just in case)
train_df["condition_text"] = pd.to_numeric(train_df["condition_text"], errors="coerce")
test_df["condition_text"] = pd.to_numeric(test_df["condition_text"], errors="coerce")

# convert all columns to numeric if needed
for col in features + [target]:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

# drop missing values
train_df = train_df.dropna()
test_df = test_df.dropna()

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

# =========================
# 4. Evaluation function
# =========================
def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(np.array(y_true) == 0, 1e-10, y_true)
    accuracy = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, accuracy

In [ ]:
scaler_sgd = StandardScaler()
X_train_sgd = scaler_sgd.fit_transform(X_train)
X_test_sgd = scaler_sgd.transform(X_test)

sgd_model = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgd_model.fit(X_train_sgd, y_train)

y_train_pred_sgd = sgd_model.predict(X_train_sgd)
y_test_pred_sgd = sgd_model.predict(X_test_sgd)

train_metrics_sgd = evaluate_model(y_train, y_train_pred_sgd)
test_metrics_sgd = evaluate_model(y_test, y_test_pred_sgd)

print("Linear (SGD) - Training Results")
print("MSE:", train_metrics_sgd[0])
print("RMSE:", train_metrics_sgd[1])
print("MAE:", train_metrics_sgd[2])
print("R2:", train_metrics_sgd[3])
print("Accuracy (%):", train_metrics_sgd[4])

print("\nLinear (SGD) - Testing Results")
print("MSE:", test_metrics_sgd[0])
print("RMSE:", test_metrics_sgd[1])
print("MAE:", test_metrics_sgd[2])
print("R2:", test_metrics_sgd[3])
print("Accuracy (%):", test_metrics_sgd[4])

Linear (SGD) - Training Results
MSE: 259.5584098392363
RMSE: 16.11081654787355
MAE: 10.073762613040227
R2: -2.314293012438416
Accuracy (%): -10767967386.659237

Linear (SGD) - Testing Results
MSE: 175.8232715895363
RMSE: 13.259836785931277
MAE: 10.51869549516511
R2: -0.41567319197349106
Accuracy (%): -28246011673.60782


In [ ]:
scaler_cnn = StandardScaler()
X_train_cnn = scaler_cnn.fit_transform(X_train)
X_test_cnn = scaler_cnn.transform(X_test)

X_train_cnn = X_train_cnn.reshape((X_train_cnn.shape[0], X_train_cnn.shape[1], 1))
X_test_cnn = X_test_cnn.reshape((X_test_cnn.shape[0], X_test_cnn.shape[1], 1))

cnn_model = Sequential([
    Input(shape=(X_train_cnn.shape[1], 1)),
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1)
])

cnn_model.compile(optimizer='adam', loss='mse')

cnn_model.fit(
    X_train_cnn,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=0
)

y_train_pred_cnn = cnn_model.predict(X_train_cnn).flatten()
y_test_pred_cnn = cnn_model.predict(X_test_cnn).flatten()

train_metrics_cnn = evaluate_model(y_train, y_train_pred_cnn)
test_metrics_cnn = evaluate_model(y_test, y_test_pred_cnn)

print("CNN - Training Results")
print("MSE:", train_metrics_cnn[0])
print("RMSE:", train_metrics_cnn[1])
print("MAE:", train_metrics_cnn[2])
print("R2:", train_metrics_cnn[3])
print("Accuracy (%):", train_metrics_cnn[4])

print("\nCNN - Testing Results")
print("MSE:", test_metrics_cnn[0])
print("RMSE:", test_metrics_cnn[1])
print("MAE:", test_metrics_cnn[2])
print("R2:", test_metrics_cnn[3])
print("Accuracy (%):", test_metrics_cnn[4])

3251/3251 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
813/813 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
CNN - Training Results
MSE: 12.611939730980243
RMSE: 3.551329290699504
MAE: 2.558125794438779
R2: 0.8389585459797975
Accuracy (%): -2038489165.3912306

CNN - Testing Results
MSE: 28.597540434436162
RMSE: 5.347666821562107
MAE: 3.698304439948562
R2: 0.7697416787697939
Accuracy (%): -13953686729.400915


In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

train_metrics_rf = evaluate_model(y_train, y_train_pred_rf)
test_metrics_rf = evaluate_model(y_test, y_test_pred_rf)

print("Random Forest - Training Results")
print("MSE:", train_metrics_rf[0])
print("RMSE:", train_metrics_rf[1])
print("MAE:", train_metrics_rf[2])
print("R2:", train_metrics_rf[3])
print("Accuracy (%):", train_metrics_rf[4])

print("\nRandom Forest - Testing Results")
print("MSE:", test_metrics_rf[0])
print("RMSE:", test_metrics_rf[1])
print("MAE:", test_metrics_rf[2])
print("R2:", test_metrics_rf[3])
print("Accuracy (%):", test_metrics_rf[4])

Random Forest - Training Results
MSE: 0.8370371344781835
RMSE: 0.9148973354853448
MAE: 0.5930702294186648
R2: 0.9893118996696321
Accuracy (%): -634083868.8601682

Random Forest - Testing Results
MSE: 27.468747287335095
RMSE: 5.241063564519619
MAE: 3.508027614322526
R2: 0.7788303630104383
Accuracy (%): -14232644927.220074


In [ ]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)

train_metrics_xgb = evaluate_model(y_train, y_train_pred_xgb)
test_metrics_xgb = evaluate_model(y_test, y_test_pred_xgb)

print("XGBoost - Training Results")
print("MSE:", train_metrics_xgb[0])
print("RMSE:", train_metrics_xgb[1])
print("MAE:", train_metrics_xgb[2])
print("R2:", train_metrics_xgb[3])
print("Accuracy (%):", train_metrics_xgb[4])

print("\nXGBoost - Testing Results")
print("MSE:", test_metrics_xgb[0])
print("RMSE:", test_metrics_xgb[1])
print("MAE:", test_metrics_xgb[2])
print("R2:", test_metrics_xgb[3])
print("Accuracy (%):", test_metrics_xgb[4])

XGBoost - Training Results
MSE: 7.121494942398675
RMSE: 2.668612924797951
MAE: 1.8937220667430783
R2: 0.9090658594328485
Accuracy (%): -1670917953.1240008

XGBoost - Testing Results
MSE: 26.214066880955052
RMSE: 5.119967468739919
MAE: 3.460173382913584
R2: 0.7889326515172366
Accuracy (%): -12145791932.986202


In [ ]:
lgbm_model = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

y_train_pred_lgbm = lgbm_model.predict(X_train)
y_test_pred_lgbm = lgbm_model.predict(X_test)

train_metrics_lgbm = evaluate_model(y_train, y_train_pred_lgbm)
test_metrics_lgbm = evaluate_model(y_test, y_test_pred_lgbm)

print("LightGBM - Training Results")
print("MSE:", train_metrics_lgbm[0])
print("RMSE:", train_metrics_lgbm[1])
print("MAE:", train_metrics_lgbm[2])
print("R2:", train_metrics_lgbm[3])
print("Accuracy (%):", train_metrics_lgbm[4])

print("\nLightGBM - Testing Results")
print("MSE:", test_metrics_lgbm[0])
print("RMSE:", test_metrics_lgbm[1])
print("MAE:", test_metrics_lgbm[2])
print("R2:", test_metrics_lgbm[3])
print("Accuracy (%):", test_metrics_lgbm[4])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009988 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2742
[LightGBM] [Info] Number of data points in the train set: 104002, number of used features: 15
[LightGBM] [Info] Start training from score 22.666454
LightGBM - Training Results
MSE: 7.639123762906873
RMSE: 2.7638964819448057
MAE: 1.9760288916121944
R2: 0.9024562736216638
Accuracy (%): -1825417449.9920006

LightGBM - Testing Results
MSE: 25.617752906252946
RMSE: 5.061398315312967
MAE: 3.4161392754040216
R2: 0.7937339824238512
Accuracy (%): -12457527658.434866


In [ ]:
# =========================
# Final Result Tables Only
# =========================

training_results_table = pd.DataFrame([
    {
        "Model": "Linear (SGD)",
        "MSE": train_metrics_sgd[0],
        "RMSE": train_metrics_sgd[1],
        "MAE": train_metrics_sgd[2],
        "R2": train_metrics_sgd[3],
        "Accuracy (%)": train_metrics_sgd[4]
    },
    {
        "Model": "CNN",
        "MSE": train_metrics_cnn[0],
        "RMSE": train_metrics_cnn[1],
        "MAE": train_metrics_cnn[2],
        "R2": train_metrics_cnn[3],
        "Accuracy (%)": train_metrics_cnn[4]
    },
    {
        "Model": "Random Forest",
        "MSE": train_metrics_rf[0],
        "RMSE": train_metrics_rf[1],
        "MAE": train_metrics_rf[2],
        "R2": train_metrics_rf[3],
        "Accuracy (%)": train_metrics_rf[4]
    },
    {
        "Model": "XGBoost",
        "MSE": train_metrics_xgb[0],
        "RMSE": train_metrics_xgb[1],
        "MAE": train_metrics_xgb[2],
        "R2": train_metrics_xgb[3],
        "Accuracy (%)": train_metrics_xgb[4]
    },
    {
        "Model": "LightGBM",
        "MSE": train_metrics_lgbm[0],
        "RMSE": train_metrics_lgbm[1],
        "MAE": train_metrics_lgbm[2],
        "R2": train_metrics_lgbm[3],
        "Accuracy (%)": train_metrics_lgbm[4]
    }
]).round(4)

testing_results_table = pd.DataFrame([
    {
        "Model": "Linear (SGD)",
        "MSE": test_metrics_sgd[0],
        "RMSE": test_metrics_sgd[1],
        "MAE": test_metrics_sgd[2],
        "R2": test_metrics_sgd[3],
        "Accuracy (%)": test_metrics_sgd[4]
    },
    {
        "Model": "CNN",
        "MSE": test_metrics_cnn[0],
        "RMSE": test_metrics_cnn[1],
        "MAE": test_metrics_cnn[2],
        "R2": test_metrics_cnn[3],
        "Accuracy (%)": test_metrics_cnn[4]
    },
    {
        "Model": "Random Forest",
        "MSE": test_metrics_rf[0],
        "RMSE": test_metrics_rf[1],
        "MAE": test_metrics_rf[2],
        "R2": test_metrics_rf[3],
        "Accuracy (%)": test_metrics_rf[4]
    },
    {
        "Model": "XGBoost",
        "MSE": test_metrics_xgb[0],
        "RMSE": test_metrics_xgb[1],
        "MAE": test_metrics_xgb[2],
        "R2": test_metrics_xgb[3],
        "Accuracy (%)": test_metrics_xgb[4]
    },
    {
        "Model": "LightGBM",
        "MSE": test_metrics_lgbm[0],
        "RMSE": test_metrics_lgbm[1],
        "MAE": test_metrics_lgbm[2],
        "R2": test_metrics_lgbm[3],
        "Accuracy (%)": test_metrics_lgbm[4]
    }
]).round(4)

print("TRAINING RESULTS TABLE")
print(training_results_table)

print("\nTESTING RESULTS TABLE")
print(testing_results_table)

TRAINING RESULTS TABLE
           Model       MSE     RMSE      MAE      R2  Accuracy (%)
0   Linear (SGD)  259.5584  16.1108  10.0738 -2.3143 -1.076797e+10
1            CNN   12.6119   3.5513   2.5581  0.8390 -2.038489e+09
2  Random Forest    0.8370   0.9149   0.5931  0.9893 -6.340839e+08
3        XGBoost    7.1215   2.6686   1.8937  0.9091 -1.670918e+09
4       LightGBM    7.6391   2.7639   1.9760  0.9025 -1.825417e+09

TESTING RESULTS TABLE
           Model       MSE     RMSE      MAE      R2  Accuracy (%)
0   Linear (SGD)  175.8233  13.2598  10.5187 -0.4157 -2.824601e+10
1            CNN   28.5975   5.3477   3.6983  0.7697 -1.395369e+10
2  Random Forest   27.4687   5.2411   3.5080  0.7788 -1.423264e+10
3        XGBoost   26.2141   5.1200   3.4602  0.7889 -1.214579e+10
4       LightGBM   25.6178   5.0614   3.4161  0.7937 -1.245753e+10


In [ ]:
display(training_results_table)
display(testing_results_table)

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Linear (SGD),259.5584,16.1108,10.0738,-2.3143,-1.076797e+10
1,CNN,12.6119,3.5513,2.5581,0.8390,-2.038489e+09
2,Random Forest,0.8370,0.9149,0.5931,0.9893,-6.340839e+08
3,XGBoost,7.1215,2.6686,1.8937,0.9091,-1.670918e+09
4,LightGBM,7.6391,2.7639,1.9760,0.9025,-1.825417e+09


,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Linear (SGD),175.8233,13.2598,10.5187,-0.4157,-2.824601e+10
1,CNN,28.5975,5.3477,3.6983,0.7697,-1.395369e+10
2,Random Forest,27.4687,5.2411,3.5080,0.7788,-1.423264e+10
3,XGBoost,26.2141,5.1200,3.4602,0.7889,-1.214579e+10
4,LightGBM,25.6178,5.0614,3.4161,0.7937,-1.245753e+10


In [ ]:
import os

print(os.listdir())

['.config', 'drive', 'sample_data']


In [ ]:
# Save both tables
training_results_table.to_csv("training_results_table.csv", index=False)
testing_results_table.to_csv("testing_results_table.csv", index=False)

# Download
from google.colab import files

files.download("training_results_table.csv")
files.download("testing_results_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>